# EV Battery Health Prediction Datasest 

Kaggel: https://www.kaggle.com/datasets/srisyra02/ev-battery-health-prediction-dataset-20k

This dataset contains 20,000 simulated electric vehicle(EV) battery telemetry records designed for binary battery-failure prediction, where battery_failure indicates whether the battery is likely to fail (1 = failure, 0 = healthy). The dataset combines battery health, charging behavior, vehicle usage, environmental conditions, maintenance and diagnostic parameters to provide a realistic foundation for battery-health analysis and predictive modeling. It's built for classification modeling, EDA, feature engineering practice and battery-health research.

In [1]:
# import libaries 
import pandas as pd
import kagglehub
from pathlib import Path


In [2]:
# Download latest version
path = kagglehub.dataset_download("srisyra02/ev-battery-health-prediction-dataset-20k")

In [3]:
# find file name 
data_folder= Path(path)

for file in data_folder.iterdir():
    print(file.name)

ev battery_failure prediction Dataset.csv


In [4]:
# assign file to data frame 
df = pd.read_csv(data_folder / "ev battery_failure prediction Dataset.csv")
df.head()

,vehicle_id,vehicle_brand,vehicle_model,vehicle_type,manufacturing_year,battery_manufacturer,battery_chemistry,battery_capacity_kwh,drive_type,odometer_km,...,sensor_fault_count,BMS_warning_count,abnormal_voltage_events,battery_stress_index,aging_score,thermal_health_score,charging_quality_score,driving_stress_score,predicted_remaining_life_cycles,battery_failure
0,EV100000,Ford,Mustang Mach-E,Truck,2022.0,Northvolt,NMC,NaN,AWD,290844.3,...,1.0,NaN,0.0,30.37,73.05,98.14,89.73,47.11,6748.0,0
1,EV100001,Tata,Tiago EV,Truck,2017.0,CATL,NCA,38.71,RWD,182239.5,...,0.0,1.0,0.0,67.94,82.50,NaN,NaN,55.38,5211.0,0
2,EV100002,Volkswagen,ID.3,Crossover,2016.0,BYD Battery,LFP,131.29,FWD,249369.3,...,0.0,1.0,0.0,74.84,90.33,60.02,86.84,74.54,4040.0,0
3,EV100003,NaN,Kona Electric,Sedan,2019.0,SK On,NMC,132.35,FWD,134830.2,...,NaN,0.0,0.0,46.94,56.69,66.43,93.93,31.69,NaN,0
4,EV100004,MG,ZS EV,SUV,2017.0,Guoxuan,LTO,25.72,AWD,158026.0,...,1.0,2.0,3.0,49.07,73.65,95.05,100.00,23.46,7290.0,0


Data set has sucessfully ingested 

## Exploritory Data Analysis 

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 70 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   vehicle_id                       20000 non-null  str    
 1   vehicle_brand                    19356 non-null  str    
 2   vehicle_model                    19180 non-null  str    
 3   vehicle_type                     19356 non-null  str    
 4   manufacturing_year               19018 non-null  float64
 5   battery_manufacturer             19382 non-null  str    
 6   battery_chemistry                19278 non-null  str    
 7   battery_capacity_kwh             19310 non-null  float64
 8   drive_type                       19274 non-null  str    
 9   odometer_km                      19134 non-null  float64
 10  vehicle_age_years                19232 non-null  float64
 11  fleet_or_private                 19106 non-null  str    
 12  battery_serial               

This dataset has 70 features. It includes a mix of float, integer, and string data types. Most variables appear to have some missing values. The target variable battery_failure and the identifier variable vehicle_id have no missing values. No more than 5% misisng for each variable. 

In [6]:
# missing summary 

missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": df.isnull().mean()*100
})

missing_summary = missing_summary[
    missing_summary["missing_count"]>0 
].sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
fast_charge_ratio,988,4.94
average_speed,984,4.92
manufacturing_year,982,4.91
home_charging_ratio,980,4.90
sensor_fault_count,968,4.84
...,...,...
battery_manufacturer,618,3.09
pack_voltage,618,3.09
average_charging_time,614,3.07
remaining_capacity,614,3.07


missingness is spread throughout the dataset between 3-5% per affected column 

In [7]:
# view missingness by row 

df["missing_count"] = df.isnull().sum(axis=1)
df["missing_percent"] = df.isnull().mean(axis=1)*100

df[["missing_count", "missing_percent"]].describe()

,missing_count,missing_percent
count,20000.000000,20000.000000
mean,2.667200,3.756620
std,1.606603,2.262821
min,0.000000,0.000000
25%,1.000000,1.408451
50%,3.000000,4.225352
75%,4.000000,5.633803
max,11.000000,15.492958


There are 20,000 observations, and each row is missing an average of 2.67 values. The median row is missing 3 values, while the most incomplete row is missing 11 values (15.5%).

missing values are spread throguht the dataset 

Imputation is likely the most approatate apprach, depending on the distrubution and datatype. 

    - Approximately symmetric continuous variable → mean
    - Skewed continuous variable or variables with outliers → median
    - Categorical/string variable → mode or possibly "Unknown" if missingness itself has meaning
    - Integer variables → depends on what the integer represents. A count like sensor_fault_count may behave differently from a coded category like 1, 2, 3.

In [8]:
df.shape

(20000, 72)

In [9]:
# drop added misisng rows 
df.drop(columns=["missing_count", "missing_percent"], inplace=True)
df.shape

(20000, 70)

## Summary 

The data set was loaded and checked for null values. Missingness was seen throughout the dataset. The next step is to check the distribution of missingness and determine imputation strategies. 